<a href="https://colab.research.google.com/github/TanishaGhosh05/FT4-NLP-PROGRAMS/blob/main/nlp_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RNN


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN,LSTM, Dense

data = """
deep learning is fun
deep learning is powerful
natural language processing is interesting
machine learning is amazing
artificial intelligence is the future
i love learning new things
learning makes life better
"""

In [ ]:



tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)


input_sequences = []

for line in data.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_len = max([len(seq) for seq in input_sequences])

input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_len, padding='pre'))


X = input_sequences[:, :-1]
y = input_sequences[:, -1]


y = tf.keras.utils.to_categorical(y, num_classes=total_words)

model = Sequential()

model.add(Embedding(total_words, 32, input_length=max_len-1))
model.add(SimpleRNN(64))
model.add(Dense(total_words, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

model.fit(X, y, epochs=200, verbose=1)

def predict_next_word(text):
    token_list = tokenizer.texts_to_sequences([text])[0]
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')

    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted)

    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word

seed_text = "deep learning is"
print("Input:", seed_text)
print("Next word:", predict_next_word(seed_text))

seed_text = "machine learning is"
print("Input:", seed_text)
print("Next word:", predict_next_word(seed_text))

LSTM

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

total_words = len(tokenizer.word_index) + 1
print("Total words:", total_words)

input_sequences = []

for line in data.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_len = max([len(seq) for seq in input_sequences])

input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_len, padding='pre')
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

model = Sequential()

model.add(Embedding(total_words, 64, input_length=max_len-1))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

model.fit(X, y, epochs=200, verbose=1)

def predict_batch(texts):
    sequences = [tokenizer.texts_to_sequences([t])[0] for t in texts]
    sequences = pad_sequences(sequences, maxlen=max_len-1, padding='pre')

    predictions = model.predict(sequences, verbose=0)

    results = []
    for pred in predictions:
        idx = np.argmax(pred)
        for word, index in tokenizer.word_index.items():
            if index == idx:
                results.append(word)
                break
    return results

test_inputs = ["deep learning is", "machine learning is"]

outputs = predict_batch(test_inputs)

for i in range(len(test_inputs)):
    print(f"Input: {test_inputs[i]}")
    print(f"Next word: {outputs[i]}")
    print()

BERT

In [ ]:
!pip install transformers

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

texts = [
    "I love this movie!",
    "This was the worst thing ever",
    "Honestly kinda mid"
]

inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

outputs = model(**inputs)
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

labels = ["NEGATIVE", "POSITIVE"]

for i, text in enumerate(texts):
    pred_label = labels[predictions[i].argmax()]
    confidence = predictions[i].max().item()
    print(f"{text} -> {pred_label} ({confidence:.4f})")

RoBERTa

In [ ]:


from transformers import RobertaTokenizer, RobertaForSequenceClassification
import torch
import torch.nn.functional as F


model_name = "cardiffnlp/twitter-roberta-base-sentiment"

tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name)

model.eval()

labels = ["Negative", "Neutral", "Positive"]

def predict_sentiment(text):

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)


    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits


    probs = F.softmax(logits, dim=1)

    predicted_class = torch.argmax(probs, dim=1).item()

    return labels[predicted_class], probs[0].tolist()

texts = [
    "this movie was amazing and beautiful",
    "this movie was terrible and boring",
    "it was okay, not great",
    "average experience nothing special"
]

for text in texts:
    label, prob = predict_sentiment(text)

    print("Text:", text)
    print("Sentiment:", label)
    print("Probabilities [Neg, Neu, Pos]:", prob)
    print()